# Stochastic Gradient Descent for (Linear) SVM — Lab Notebook (Primal Formulation)

This notebook follows the lab handout **“Stochastic gradient descent for SVM”** and implements the **three proposed experiments**:
1) classical SGD (1 sample / update),  
2) mini-batch SGD,  
3) comparison of convergence speed using **log f(w,b)** per epoch for SGD vs mini-batch vs full gradient descent.

We work with the **primal, unconstrained** hinge-loss objective:

\begin{aligned}
f(w,b) &= \frac{\lambda}{2}\|w\|^2 + \sum_{i=1}^{m}\max\big(0,\, 1 - y_i(w^\top x_i + b)\big).
\end{aligned}

(See the lab PDF.) fileciteturn0file0


## Theoretical background

### 1) From constrained soft-margin to unconstrained hinge-loss

For non-separable data, the soft-margin primal problem introduces slack variables \(\xi_i\ge 0\) and a penalty \(K\sum_i \xi_i\).  
Eliminating \(\xi\) leads to the hinge-loss formulation:

\[
\min_{w,b}\ \frac{1}{2}\|w\|^2 + K\sum_{i=1}^{m}\max\left(0,\, 1 - y_i(w^\top x_i + b)\right).
\]

The lab handout uses a numerically friendlier scaling:

\[
f(w,b)=\frac{\lambda}{2}\|w\|^2 + \sum_{i=1}^{m}\max\left(0,\, 1 - y_i(w^\top x_i + b)\right),
\]
where typically \(\lambda\) is “small” and relates to the usual \(K\) (or \(C\) in ML libraries) by approximately \(\lambda \approx 1/K\). fileciteturn0file0

### 2) Subgradients of the hinge loss

Define the margin for sample \(i\):
\[
s_i = y_i(w^\top x_i + b).
\]
The hinge term is \(h_i(w,b)=\max(0,1-s_i)\).

- If \(s_i > 1\): the hinge is inactive, subgradient is \(0\).
- If \(s_i < 1\): the hinge is active and
  \[
  \partial_w h_i = -y_i x_i,\quad \partial_b h_i = -y_i.
  \]
- If \(s_i = 1\): any convex combination between \(0\) and the active case is valid (we can safely pick the “active” formula in code).

Thus a valid subgradient for the full objective is:
\[
g_w = \lambda w + \sum_{i\in \mathcal{A}}(-y_i x_i),\quad
g_b = \sum_{i\in \mathcal{A}}(-y_i),
\]
where \(\mathcal{A}=\{i: y_i(w^\top x_i+b) < 1\}\) is the active set.

### 3) Why stochastic / mini-batch works here

Computing the exact gradient/subgradient over all \(m\) points each step can be expensive. SGD replaces that sum by an unbiased estimate using:
- **1 sample** (classical SGD), or
- **M samples** (mini-batch), which reduces noise.

The lab recommends decreasing learning rates \(\gamma_t\) (e.g. \(\gamma_t \sim 1/t\)), implemented via \(\gamma_t=1/\mu_t\) with \(\mu_t\) increasing. fileciteturn0file0

### 4) What we will measure

- The decision boundary \(w^\top x + b = 0\) and margin lines \(w^\top x + b = \pm 1\).
- The objective value \(f(w,b)\) after each epoch.
- The **logarithm** \(\log f(w,b)\) per epoch to compare convergence speed as requested. fileciteturn0file0


In [ ]:
# --- Imports & utility configuration ---
import numpy as np
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Tuple

# Optional: for a reference solver (often implemented via dual / SMO internally)
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

np.random.seed(42)


## Data generation (2D, mostly separable)

The handout suggests using **separable sets** to visualize margins clearly. We generate two Gaussian blobs in 2D, and optionally add overlap.


In [ ]:
def make_blobs_2d(n_per_class=100, sep=2.5, noise=0.8, overlap=False):
    '''
    Create a simple 2D binary classification dataset.
    y in {-1, +1}
    '''
    n = n_per_class
    # Centers
    c1 = np.array([-sep, 0.0])
    c2 = np.array([+sep, 0.0])
    X_pos = c2 + noise * np.random.randn(n, 2)
    X_neg = c1 + noise * np.random.randn(n, 2)

    X = np.vstack([X_neg, X_pos])
    y = np.hstack([-np.ones(n), +np.ones(n)])

    if overlap:
        # reduce separation to make it less separable
        X_pos2 = np.array([+sep*0.6, 0.3]) + noise*1.1*np.random.randn(n,2)
        X_neg2 = np.array([-sep*0.6, -0.3]) + noise*1.1*np.random.randn(n,2)
        X = np.vstack([X_neg2, X_pos2])
        y = np.hstack([-np.ones(n), +np.ones(n)])

    # Shuffle
    idx = np.random.permutation(len(y))
    return X[idx], y[idx]

X, y = make_blobs_2d(n_per_class=100, sep=2.5, noise=0.7, overlap=False)

plt.figure()
plt.scatter(X[y==-1,0], X[y==-1,1], label="Class -1")
plt.scatter(X[y==+1,0], X[y==+1,1], label="Class +1")
plt.title("Training data")
plt.legend()
plt.show()

print("X shape:", X.shape, "y shape:", y.shape, "classes:", np.unique(y))


## Core implementation (primal objective + SGD variants)

We implement:
- the primal objective \(f(w,b)\),
- full-batch subgradient (gradient descent),
- classical SGD (one sample per update, with shuffling + epochs),
- mini-batch SGD,
- and plotting utilities for the decision boundary and margins.


In [ ]:
def svm_primal_objective(w: np.ndarray, b: float, X: np.ndarray, y: np.ndarray, lam: float) -> float:
    '''f(w,b) = lam/2 * ||w||^2 + sum_i max(0, 1 - y_i (w^T x_i + b))'''
    margins = y * (X @ w + b)
    hinge = np.maximum(0.0, 1.0 - margins)
    return 0.5 * lam * float(w @ w) + float(np.sum(hinge))

def subgrad_full(w: np.ndarray, b: float, X: np.ndarray, y: np.ndarray, lam: float) -> Tuple[np.ndarray, float]:
    '''Full subgradient of the objective over all samples.'''
    margins = y * (X @ w + b)
    active = margins < 1.0
    grad_w = lam * w - (y[active, None] * X[active]).sum(axis=0)
    grad_b = -y[active].sum()
    return grad_w, float(grad_b)

def subgrad_batch(w: np.ndarray, b: float, Xb: np.ndarray, yb: np.ndarray, lam: float) -> Tuple[np.ndarray, float]:
    '''Mini-batch subgradient.'''
    margins = yb * (Xb @ w + b)
    active = margins < 1.0
    grad_w = lam * w - (yb[active, None] * Xb[active]).sum(axis=0)
    grad_b = -yb[active].sum()
    return grad_w, float(grad_b)

def predict(w: np.ndarray, b: float, X: np.ndarray) -> np.ndarray:
    return np.where((X @ w + b) >= 0.0, 1.0, -1.0)

def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(y_true == y_pred))

def plot_separator_and_margins(w: np.ndarray, b: float, X: np.ndarray, y: np.ndarray, title: str = ""):
    plt.figure()
    plt.scatter(X[y==-1,0], X[y==-1,1], label="Class -1")
    plt.scatter(X[y==+1,0], X[y==+1,1], label="Class +1")

    x_min, x_max = X[:,0].min()-1.0, X[:,0].max()+1.0
    xs = np.linspace(x_min, x_max, 200)

    if abs(w[1]) < 1e-12:
        x0 = (-b) / (w[0] + 1e-12)
        plt.axvline(x0, linestyle="-", label="decision (0)")
    else:
        for c, ls, lab in [(0.0, "-", "decision (0)"), (1.0, "--", "+1 margin"), (-1.0, "--", "-1 margin")]:
            ys = (c - b - w[0]*xs) / w[1]
            plt.plot(xs, ys, linestyle=ls, label=lab)

    plt.title(title)
    plt.legend()
    plt.show()


## Proposed experiments (from the handout)

We follow the three experiments described in the lab:

1. **Classical SGD**: shuffle samples and iterate one-by-one (epoch). Suggested \(\gamma_t = 1/\mu_t\) with \(\mu_0 \approx 100\) and increasing \(\mu\); suggested \(\lambda \in \{10^{-4},10^{-2}\}\). fileciteturn0file0  
2. **Mini-batch**: \(m=200\) samples and batch sizes like 10, 20, 50; reshuffle each epoch and rebuild batches. fileciteturn0file0  
3. **Compare log objective**: plot \(\log f(w,b)\) per epoch for SGD, mini-batch, and full gradient descent (batch size \(=m\)). fileciteturn0file0  


### Experiment 1 — Classical SGD (1 sample per update)

- We run for a fixed number of epochs.
- Inside each epoch: shuffle indices and do one update per sample.
- Learning rate: γ_t = 1/μ_t with μ increased by `mu_inc` after each update.
- Record f(w,b) after each epoch.


In [ ]:
@dataclass
class TrainConfig:
    lam: float = 1e-2
    epochs: int = 60
    mu0: float = 100.0
    mu_inc: float = 0.5
    w0_scale: float = 0.0
    b0: float = 0.0
    verbose: bool = True

def train_sgd_one_sample(X: np.ndarray, y: np.ndarray, cfg: TrainConfig):
    n_features = X.shape[1]
    w = cfg.w0_scale * np.random.randn(n_features)
    b = float(cfg.b0)
    mu = float(cfg.mu0)

    f_hist, acc_hist = [], []

    for ep in range(cfg.epochs):
        idx = np.random.permutation(len(y))
        for i in idx:
            gamma = 1.0 / mu
            xi, yi = X[i], y[i]
            margin = yi * (w @ xi + b)

            if margin > 1.0:
                grad_w = cfg.lam * w
                grad_b = 0.0
            else:
                grad_w = cfg.lam * w - yi * xi
                grad_b = -yi

            w = w - gamma * grad_w
            b = b - gamma * grad_b
            mu += cfg.mu_inc

        fval = svm_primal_objective(w, b, X, y, cfg.lam)
        acc = accuracy(y, predict(w, b, X))
        f_hist.append(fval)
        acc_hist.append(acc)

        if cfg.verbose and (ep % max(1, cfg.epochs//6) == 0 or ep == cfg.epochs-1):
            print(f"[SGD-1] epoch {ep+1:3d}/{cfg.epochs}  f={fval:.4f}  acc={acc:.3f}  mu={mu:.1f}")

    return w, b, np.array(f_hist), np.array(acc_hist)

cfg1 = TrainConfig(lam=1e-2, epochs=60, mu0=100.0, mu_inc=0.5, verbose=True)
w_sgd1, b_sgd1, f_sgd1, acc_sgd1 = train_sgd_one_sample(X, y, cfg1)

plot_separator_and_margins(w_sgd1, b_sgd1, X, y, title="Experiment 1 — SGD (1 sample/update)")
print("Final training accuracy:", acc_sgd1[-1])


### Experiment 2 — Mini-batch SGD

We keep m=200 samples and try batch sizes 10, 20, 50.
Each epoch:
- shuffle data,
- split into mini-batches,
- update per batch,
- record f(w,b).


In [ ]:
def train_minibatch(X: np.ndarray, y: np.ndarray, batch_size: int, cfg: TrainConfig):
    n_features = X.shape[1]
    w = cfg.w0_scale * np.random.randn(n_features)
    b = float(cfg.b0)
    mu = float(cfg.mu0)

    f_hist, acc_hist = [], []
    m = len(y)

    for ep in range(cfg.epochs):
        idx = np.random.permutation(m)
        Xs, ys = X[idx], y[idx]

        for start in range(0, m, batch_size):
            end = min(m, start + batch_size)
            Xb, yb = Xs[start:end], ys[start:end]

            gamma = 1.0 / mu
            grad_w, grad_b = subgrad_batch(w, b, Xb, yb, cfg.lam)
            w = w - gamma * grad_w
            b = b - gamma * grad_b
            mu += cfg.mu_inc

        fval = svm_primal_objective(w, b, X, y, cfg.lam)
        acc = accuracy(y, predict(w, b, X))
        f_hist.append(fval)
        acc_hist.append(acc)

    return w, b, np.array(f_hist), np.array(acc_hist)

cfg2 = TrainConfig(lam=1e-2, epochs=60, mu0=100.0, mu_inc=0.5, verbose=False)
results_mb = {}

for bs in [10, 20, 50]:
    w_mb, b_mb, f_mb, acc_mb = train_minibatch(X, y, batch_size=bs, cfg=cfg2)
    results_mb[bs] = (w_mb, b_mb, f_mb, acc_mb)

for bs, (w_mb, b_mb, f_mb, acc_mb) in results_mb.items():
    plot_separator_and_margins(w_mb, b_mb, X, y, title=f"Experiment 2 — Mini-batch size = {bs}")
    print(f"Batch {bs:02d}: final f={f_mb[-1]:.4f}, final acc={acc_mb[-1]:.3f}")


### Experiment 3 — Compare convergence: \(\log f(w,b)\) vs epoch

We compare:
- SGD (1 sample/update),
- mini-batch (choose 20),
- full gradient descent (batch size = all samples).

We plot **log f(w,b)** per epoch, as requested. fileciteturn0file0


In [ ]:
def train_full_batch_gd(X: np.ndarray, y: np.ndarray, cfg: TrainConfig):
    n_features = X.shape[1]
    w = cfg.w0_scale * np.random.randn(n_features)
    b = float(cfg.b0)
    mu = float(cfg.mu0)

    f_hist, acc_hist = [], []

    for ep in range(cfg.epochs):
        gamma = 1.0 / mu
        grad_w, grad_b = subgrad_full(w, b, X, y, cfg.lam)
        w = w - gamma * grad_w
        b = b - gamma * grad_b
        mu += cfg.mu_inc  # here iteration = epoch

        fval = svm_primal_objective(w, b, X, y, cfg.lam)
        acc = accuracy(y, predict(w, b, X))
        f_hist.append(fval)
        acc_hist.append(acc)

    return w, b, np.array(f_hist), np.array(acc_hist)

# Train the three methods
w_sgd, b_sgd, f_sgd, acc_sgd = train_sgd_one_sample(X, y, TrainConfig(lam=1e-2, epochs=60, mu0=100.0, mu_inc=0.5, verbose=False))
w_mb20, b_mb20, f_mb20, acc_mb20 = train_minibatch(X, y, batch_size=20, cfg=TrainConfig(lam=1e-2, epochs=60, mu0=100.0, mu_inc=0.5, verbose=False))
w_gd, b_gd, f_gd, acc_gd = train_full_batch_gd(X, y, TrainConfig(lam=1e-2, epochs=60, mu0=100.0, mu_inc=5.0, verbose=False))

# Plot log objective per epoch
eps = 1e-12
plt.figure()
plt.plot(np.log(f_sgd + eps), label="SGD (1 sample)")
plt.plot(np.log(f_mb20 + eps), label="Mini-batch (20)")
plt.plot(np.log(f_gd + eps), label="Full-batch GD")
plt.xlabel("Epoch")
plt.ylabel("log f(w,b)")
plt.title("Experiment 3 — log objective vs epoch")
plt.legend()
plt.show()

print("Final acc: SGD", acc_sgd[-1], "| MB20", acc_mb20[-1], "| GD", acc_gd[-1])


## (Optional) Comparison to a library “dual-style” solver

The handout asks to compare with the **dual method** from the previous lab. If you used a linear SVM there, you can use a library fit as a reference.

Many libraries use:
\(\frac{1}{2}\|w\|^2 + C\sum_i \max(0,1-y_i(w^\top x_i + b))\).  
Our formulation is:
\(\frac{\lambda}{2}\|w\|^2 + \sum_i \max(0,1-y_i(w^\top x_i + b))\).  
So a rough correspondence is \(C \approx 1/\lambda\). fileciteturn0file0


In [ ]:
lam_ref = 1e-2
C_ref = 1.0 / lam_ref

# Fit linear SVC reference with scaling (common good practice)
clf = make_pipeline(StandardScaler(), SVC(kernel="linear", C=C_ref))
clf.fit(X, y)

# Extract w,b in original coordinates
svc = clf.named_steps['svc']
scaler = clf.named_steps['standardscaler']

w_s = svc.coef_.ravel()
b_s = float(svc.intercept_[0])
w_lib = w_s / scaler.scale_
b_lib = b_s - float((w_s / scaler.scale_) @ scaler.mean_)

plot_separator_and_margins(w_lib, b_lib, X, y, title=f"Reference SVC linear (C={C_ref:.1f})")

# Our primal SGD with matching lambda
w_pr, b_pr, f_pr, acc_pr = train_sgd_one_sample(X, y, TrainConfig(lam=lam_ref, epochs=60, mu0=100.0, mu_inc=0.5, verbose=False))
plot_separator_and_margins(w_pr, b_pr, X, y, title=f"Primal SGD (lambda={lam_ref:g})")

print("Training accuracy (SVC):", accuracy(y, predict(w_lib, b_lib, X)))
print("Training accuracy (primal SGD):", accuracy(y, predict(w_pr, b_pr, X)))


## Conclusions

- The **primal hinge-loss** formulation trains an SVM by minimizing \(f(w,b)\) directly (no explicit constraints/slacks needed in code). fileciteturn0file0  
- **SGD (1 sample/update)** often improves quickly early on but exhibits noisier progress in the objective.
- **Mini-batches** reduce gradient noise and commonly produce smoother convergence curves.
- **Full-batch gradient descent** gives the smoothest \(\log f(w,b)\) trajectory per epoch but uses all samples each update.
- Plotting \(\log f(w,b)\) per epoch is a clean way to compare convergence speed across methods, exactly as requested. fileciteturn0file0  
